# Nautiq — Gold de predicciones JIT actuales

**Granularidad:** 1 fila = 1 escala portuaria activa por buque.

La detección de `SERVICE`, `ANCHOR`, `port_call` y el snapshot AIS utiliza la misma lógica que `vessel_port_calls_analytics`.

Se conservan únicamente buques que:
- tienen destino `ESVLC`, `ESBCN` o `ESALG`;
- siguen dentro de la escala portuaria actual;
- todavía no tienen `service_start_timestamp`;
- están aproximándose o ya han iniciado fondeo.

El modelo registrado utiliza solo:
`speed_over_ground_knots`, `navigation_status_code` y `destination_port_code`.


In [0]:
%run ../setup_nautiq_dev


# Nautiq - setup del entorno

Configuración centralizada para el flujo activo del TFM:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado técnico de Auto Loader.
- Tablas Silver DEV.
- Gold histórico de port calls.
- Gold de baseline histórico de espera.
- Gold de predicciones JIT actuales.
- Modelo ML registrado en Unity Catalog con alias `Champion`.
- Cambio futuro entre tablas administradas y ADLS externo.

### Notebooks activos

- `silver_ais_positions_dev`
- `silver_ais_static_dev`
- `gold_vessel_port_calls_jit`
- `gold_waiting_avg_per_length`
- `ml_vessel_jit_classification`
- `gold_vessel_jit_current_predictions`


DataFrame[]

NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 17 elementos encontrados
[OK] ais_static: 25 elementos encontrados

Silver DEV tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold tables:
  - masterxyz002dbr.gold.vessel_port_calls_analytics
  - masterxyz002dbr.gold.waiting_avg_per_length
  - masterxyz002dbr.gold.vessel_jit_current_predictions

Registered ML model:
  - masterxyz002dbr.gold.vessel_jit_classifier@Champion

Active notebooks:
  - silver_ais_positions_dev
  - silver_ais_static_dev
  - gold_vessel_port_calls_jit
  - gold_waiting_avg_per_length
  - ml_vessel_jit_classification
  - gold_vessel_jit_current_predictions

[OK] Setup completado correctamente.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from mlflow import MlflowClient
import mlflow, mlflow.sklearn
import pandas as pd
import json

spark.conf.set("spark.sql.session.timeZone", "UTC")

positions_table = positions_target_table
static_table = static_target_table
baseline_table = waiting_avg_per_length_target_table

target_table = vessel_jit_predictions_target_table
target_path = vessel_jit_predictions_target_path

mlflow.set_registry_uri("databricks-uc")
REGISTERED_MODEL_NAME = registered_model_name
MODEL_URI = registered_model_uri


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

print("Positions:", positions_table)
print("Static:", static_table)
print("Baseline:", baseline_table)
print("Modelo:", MODEL_URI)
print("Gold:", target_table)


Positions: masterxyz002dbr.silver.ais_positions_dev
Static: masterxyz002dbr.silver.ais_static_dev
Baseline: masterxyz002dbr.gold.waiting_avg_per_length
Modelo: models:/masterxyz002dbr.gold.vessel_jit_classifier@Champion
Gold: masterxyz002dbr.gold.vessel_jit_current_predictions


In [0]:
# ============================================================
# 1. PARAMETROS — IGUALES AL GOLD BASE
# ============================================================

EARTH_RADIUS_NM = 3440.065
APPROACH_RADIUS_NM = 25.0
SERVICE_RADIUS_NM = 0.5
ANCHOR_RADIUS_NM = 5.0

SERVICE_MAX_SPEED_KNOTS = 0.1
ANCHOR_MAX_SPEED_KNOTS = 0.5
APPROACH_MIN_SPEED_KNOTS = 1.0

MIN_SERVICE_MINUTES = 10.0
MIN_ANCHOR_MINUTES = 10.0
MIN_SEGMENT_POSITIONS = 2
MAX_CONTINUOUS_GAP_MINUTES = 30.0
PORT_CALL_GAP_HOURS = 12.0

MIN_DISTANCE_PROGRESS_NM = 0.02
MAX_COG_SERVICE_DIFF_DEG = 60.0

# La configuración de los puertos se carga desde setup_nautiq_dev.
ports_schema = StructType([
    StructField("destination_port_code", StringType(), False),
    StructField("destination_port_name_cfg", StringType(), False),
    StructField("service_latitude", DoubleType(), False),
    StructField("service_longitude", DoubleType(), False),
])

ports_df = spark.createDataFrame(ports_data, ports_schema)


## 2. Silver + Static vigente

La lectura replica el Gold base: mismo `bronze_start_date`, deduplicación Kafka y `AS-OF JOIN` de Static según el timestamp de cada posición.


In [0]:
required_tables = [positions_table, static_table, baseline_table]
missing_tables = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing_tables: raise RuntimeError("Faltan tablas necesarias: " + ", ".join(missing_tables))

start_date = F.to_date(F.lit(bronze_start_date))
positions = spark.table(positions_table).filter(F.to_date("bronze_partition_date") >= start_date).filter(F.col("event_timestamp").isNotNull()).filter(F.col("latitude").between(-90.0, 90.0)).filter(F.col("longitude").between(-180.0, 180.0)).dropDuplicates(["kafka_partition", "kafka_offset"])
static = spark.table(static_table).filter(F.to_date("bronze_partition_date") >= start_date).dropDuplicates(["kafka_partition", "kafka_offset"])

static_time = static.withColumn("static_valid_from", F.coalesce("kafka_ingestion_timestamp", "bronze_ingested_timestamp", "silver_processed_timestamp")).filter(F.col("static_valid_from").isNotNull())
w_static_ts = Window.partitionBy("mmsi", "static_valid_from").orderBy(F.col("kafka_offset").desc_nulls_last())
static_time = static_time.withColumn("_rn", F.row_number().over(w_static_ts)).filter(F.col("_rn") == 1).drop("_rn")
w_static = Window.partitionBy("mmsi").orderBy("static_valid_from", "kafka_offset")
static_intervals = static_time.withColumn("static_valid_to", F.lead("static_valid_from").over(w_static))

p, s = positions.alias("p"), static_intervals.alias("s")
join_condition = (F.col("p.mmsi") == F.col("s.mmsi")) & (F.col("p.event_timestamp") >= F.col("s.static_valid_from")) & (F.col("s.static_valid_to").isNull() | (F.col("p.event_timestamp") < F.col("s.static_valid_to")))

enriched = p.join(s, join_condition, "left").select(
    F.col("p.mmsi").alias("mmsi"), F.col("p.event_timestamp").alias("event_timestamp"),
    F.col("p.latitude").alias("latitude"), F.col("p.longitude").alias("longitude"),
    F.col("p.speed_over_ground_knots").alias("speed_over_ground_knots"),
    F.col("p.course_over_ground_degrees").alias("course_over_ground_degrees"),
    F.col("p.true_heading_degrees").alias("true_heading_degrees"),
    F.col("p.navigation_status_code").alias("navigation_status_code"),
    F.col("s.imo").alias("imo"), F.col("s.ship_type_code").alias("ship_type_code"),
    F.col("s.vessel_length_meters").alias("vessel_length_meters"),
    F.col("s.destination_port_code").alias("destination_port_code"),
).join(F.broadcast(ports_df), "destination_port_code", "inner")


## 3. Port call y aproximación

Se reutilizan los mismos radios, progreso, rumbo y separación de escalas de 12 horas que en el Gold histórico.


In [0]:
lat1, lon1 = F.radians("latitude"), F.radians("longitude")
lat2, lon2 = F.radians("service_latitude"), F.radians("service_longitude")
dlat, dlon = lat2 - lat1, lon2 - lon1
a = F.pow(F.sin(dlat / 2.0), 2) + F.cos(lat1) * F.cos(lat2) * F.pow(F.sin(dlon / 2.0), 2)
distance_nm = F.lit(EARTH_RADIUS_NM) * 2.0 * F.asin(F.sqrt(F.least(F.lit(1.0), F.greatest(F.lit(0.0), a))))
bearing_deg = F.pmod(F.degrees(F.atan2(F.sin(dlon) * F.cos(lat2), F.cos(lat1) * F.sin(lat2) - F.sin(lat1) * F.cos(lat2) * F.cos(dlon))) + 360.0, 360.0)
cog_diff_deg = F.abs(F.pmod(F.col("course_over_ground_degrees") - bearing_deg + 180.0, 360.0) - 180.0)

geo = enriched.withColumn("distance_to_service_nm", distance_nm).withColumn("bearing_to_service_degrees", bearing_deg).withColumn("cog_to_service_difference_degrees", F.when(F.col("course_over_ground_degrees").between(0.0, 360.0), cog_diff_deg))

w_track = Window.partitionBy("mmsi", "destination_port_code").orderBy("event_timestamp")
track = geo.withColumn("previous_distance_nm", F.lag("distance_to_service_nm").over(w_track)).withColumn("previous_event_timestamp", F.lag("event_timestamp").over(w_track))
track = track.withColumn("distance_progress_nm", F.col("previous_distance_nm") - F.col("distance_to_service_nm")).withColumn("gap_from_previous_hours", (F.unix_timestamp("event_timestamp") - F.unix_timestamp("previous_event_timestamp")) / 3600.0)
track = track.withColumn("inside_approach_zone", F.col("distance_to_service_nm") <= APPROACH_RADIUS_NM)
track = track.withColumn("approach_signal", (F.col("inside_approach_zone") & (F.col("speed_over_ground_knots") > APPROACH_MIN_SPEED_KNOTS) & (F.col("distance_progress_nm") >= MIN_DISTANCE_PROGRESS_NM) & (F.col("cog_to_service_difference_degrees") <= MAX_COG_SERVICE_DIFF_DEG)).cast("int"))
track = track.withColumn("previous_inside_approach_zone", F.lag("inside_approach_zone").over(w_track))
track = track.withColumn("new_port_call_flag", F.when(F.col("inside_approach_zone") & (F.col("previous_inside_approach_zone").isNull() | (~F.col("previous_inside_approach_zone")) | (F.col("gap_from_previous_hours") > PORT_CALL_GAP_HOURS)), 1).otherwise(0))
track = track.withColumn("port_call_sequence", F.sum("new_port_call_flag").over(w_track.rowsBetween(Window.unboundedPreceding, Window.currentRow)))

calls_positions = track.filter(F.col("inside_approach_zone") & (F.col("port_call_sequence") > 0))
keys = ["mmsi", "destination_port_code", "port_call_sequence"]
w_call = Window.partitionBy(*keys).orderBy("event_timestamp")
w_call_cum = w_call.rowsBetween(Window.unboundedPreceding, Window.currentRow)


## 4. Primer SERVICE y primer ANCHOR

La detección es la misma que en `vessel_port_calls_analytics`:
- `SERVICE`: `<= 0.5 NM`, `SOG <= 0.1 kn` y al menos 10 minutos.
- `ANCHOR`: antes del primer servicio, `SOG <= 0.5 kn`, dentro de 5 NM y al menos 10 minutos.

Por ello `anchor_start_timestamp` tendrá el mismo significado que en el Gold base.


In [0]:
service_rows = calls_positions.withColumn("_service_signal", ((F.col("distance_to_service_nm") <= SERVICE_RADIUS_NM) & (F.col("speed_over_ground_knots") <= SERVICE_MAX_SPEED_KNOTS) & (F.col("navigation_status_code").isNull() | (F.col("navigation_status_code") != 1))).cast("int"))
service_rows = service_rows.withColumn("_prev_service_signal", F.lag("_service_signal").over(w_call)).withColumn("_next_service_signal", F.lead("_service_signal").over(w_call)).withColumn("_next_timestamp", F.lead("event_timestamp").over(w_call))
service_rows = service_rows.withColumn("_new_service_segment", F.when((F.col("_service_signal") == 1) & (F.coalesce(F.col("_prev_service_signal"), F.lit(0)) != 1), 1).otherwise(0))
service_rows = service_rows.withColumn("_service_segment_id", F.sum("_new_service_segment").over(w_call_cum))
service_rows = service_rows.withColumn("_service_interval_seconds", F.when((F.col("_service_signal") == 1) & (F.col("_next_service_signal") == 1) & ((F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).between(0, MAX_CONTINUOUS_GAP_MINUTES * 60)), F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).otherwise(0))

service_segments = service_rows.filter(F.col("_service_signal") == 1).groupBy(*keys, "_service_segment_id").agg(F.min("event_timestamp").alias("service_start_timestamp"), F.max("event_timestamp").alias("service_last_timestamp"), (F.sum("_service_interval_seconds") / 60.0).alias("service_observed_minutes"), F.count("*").alias("service_position_count"))
service_segments = service_segments.filter((F.col("service_observed_minutes") >= MIN_SERVICE_MINUTES) & (F.col("service_position_count") >= MIN_SEGMENT_POSITIONS))
w_first_service = Window.partitionBy(*keys).orderBy("service_start_timestamp")
first_service = service_segments.withColumn("_rn", F.row_number().over(w_first_service)).filter(F.col("_rn") == 1).select(*keys, "service_start_timestamp")

before_service = calls_positions.join(first_service, keys, "left").filter(F.col("service_start_timestamp").isNull() | (F.col("event_timestamp") < F.col("service_start_timestamp")))
before_service = before_service.withColumn("_anchor_method", F.when((F.col("distance_to_service_nm") <= ANCHOR_RADIUS_NM) & (F.col("speed_over_ground_knots") <= ANCHOR_MAX_SPEED_KNOTS) & (F.col("navigation_status_code") == 1), "STATUS_1").when((F.col("distance_to_service_nm") > SERVICE_RADIUS_NM) & (F.col("distance_to_service_nm") <= ANCHOR_RADIUS_NM) & (F.col("speed_over_ground_knots") <= ANCHOR_MAX_SPEED_KNOTS) & (F.col("navigation_status_code").isNull() | (~F.col("navigation_status_code").isin(1, 5))), "LOW_SPEED"))
before_service = before_service.withColumn("_anchor_signal", F.col("_anchor_method").isNotNull().cast("int"))
before_service = before_service.withColumn("_prev_anchor_signal", F.lag("_anchor_signal").over(w_call)).withColumn("_next_anchor_signal", F.lead("_anchor_signal").over(w_call)).withColumn("_next_timestamp", F.lead("event_timestamp").over(w_call))
before_service = before_service.withColumn("_new_anchor_segment", F.when((F.col("_anchor_signal") == 1) & (F.coalesce(F.col("_prev_anchor_signal"), F.lit(0)) != 1), 1).otherwise(0))
before_service = before_service.withColumn("_anchor_segment_id", F.sum("_new_anchor_segment").over(w_call_cum))
before_service = before_service.withColumn("_anchor_interval_seconds", F.when((F.col("_anchor_signal") == 1) & (F.col("_next_anchor_signal") == 1) & ((F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).between(0, MAX_CONTINUOUS_GAP_MINUTES * 60)), F.unix_timestamp("_next_timestamp") - F.unix_timestamp("event_timestamp")).otherwise(0))

anchor_segments = before_service.filter(F.col("_anchor_signal") == 1).groupBy(*keys, "_anchor_segment_id").agg(F.min("event_timestamp").alias("anchor_start_timestamp"), F.max("event_timestamp").alias("anchor_last_timestamp"), (F.sum("_anchor_interval_seconds") / 60.0).alias("anchor_observed_minutes"), F.count("*").alias("anchor_position_count"))
anchor_segments = anchor_segments.filter((F.col("anchor_observed_minutes") >= MIN_ANCHOR_MINUTES) & (F.col("anchor_position_count") >= MIN_SEGMENT_POSITIONS))
w_first_anchor = Window.partitionBy(*keys).orderBy("anchor_start_timestamp")
first_anchor = anchor_segments.withColumn("_rn", F.row_number().over(w_first_anchor)).filter(F.col("_rn") == 1).select(*keys, "anchor_start_timestamp")


## 5. Snapshot previo a la operación

Igual que en el Gold base, `event_timestamp` y las variables AIS guardadas representan la última posición anterior a `ANCHOR` o `SERVICE`. Si todavía no ha ocurrido ninguno, representan la última posición de la escala.


In [0]:
call_state = first_service.join(first_anchor, keys, "full")
cutoff = call_state.select(*keys, F.coalesce("anchor_start_timestamp", "service_start_timestamp").alias("_operation_start_timestamp"))
before_operation = calls_positions.join(cutoff, keys, "left").filter(F.col("_operation_start_timestamp").isNull() | (F.col("event_timestamp") < F.col("_operation_start_timestamp")))

approach_summary = before_operation.groupBy(*keys).agg(F.max("approach_signal").alias("has_approach_evidence"))
w_pre = Window.partitionBy(*keys).orderBy(F.col("event_timestamp").desc())
feature_snapshot = before_operation.withColumn("_rn", F.row_number().over(w_pre)).filter(F.col("_rn") == 1).select(*keys, "event_timestamp", "speed_over_ground_knots", "course_over_ground_degrees", "true_heading_degrees", "navigation_status_code", "distance_to_service_nm")

call_summary = calls_positions.groupBy(*keys).agg(
    F.min("event_timestamp").alias("port_call_start_timestamp"),
    F.max("event_timestamp").alias("port_call_end_timestamp"),
    F.first("imo", ignorenulls=True).alias("imo"),
    F.first("ship_type_code", ignorenulls=True).alias("ship_type_code"),
    F.first("vessel_length_meters", ignorenulls=True).alias("vessel_length_meters"),
)

calls = call_summary.join(call_state, keys, "left").join(approach_summary, keys, "left").join(feature_snapshot, keys, "left")

ship_code = F.col("ship_type_code")
calls = calls.withColumn("ship_type_category", F.when(ship_code.between(20, 29), "WIG").when(ship_code == 30, "Pesca").when(ship_code.isin(31, 32), "Remolque").when(ship_code == 33, "Dragado / operaciones submarinas").when(ship_code == 34, "Operaciones de buceo").when(ship_code == 35, "Operaciones militares").when(ship_code == 36, "Vela").when(ship_code == 37, "Recreo").when(ship_code.between(40, 49), "Alta velocidad").when(ship_code == 50, "Practico").when(ship_code == 51, "Salvamento y rescate").when(ship_code == 52, "Remolcador").when(ship_code == 53, "Servicio portuario").when(ship_code == 54, "Anticontaminacion").when(ship_code == 55, "Fuerzas del orden").when(ship_code == 58, "Transporte medico").when(ship_code == 59, "No combatiente").when(ship_code.between(60, 69), "Pasaje").when(ship_code.between(70, 79), "Carga").when(ship_code.between(80, 89), "Tanque").when(ship_code.between(90, 99), "Otros").when(ship_code == 0, "No disponible").otherwise("Reservado"))

length_m = F.col("vessel_length_meters")
calls = calls.withColumn("vessel_length_band", F.when((length_m > 0) & (length_m < 100), "0-100").when((length_m >= 100) & (length_m < 200), "100-200").when((length_m >= 200) & (length_m < 300), "200-300").when((length_m >= 300) & (length_m <= 600), "300-600").otherwise("OTHER"))

imo_long = F.col("imo").cast("long")
calls = calls.withColumn("vessel_id", F.when(imo_long.isNotNull() & (imo_long > 0), F.concat(F.lit("IMO_"), imo_long.cast("string"))).otherwise(F.concat(F.lit("MMSI_"), F.col("mmsi").cast("string"))))
calls = calls.withColumn("port_call_id", F.sha2(F.concat_ws("|", "vessel_id", "destination_port_code", F.date_format("port_call_start_timestamp", "yyyy-MM-dd HH:mm:ss.SSSSSS")), 256))


## 6. Escala activa actual

Se compara la escala con la última posición real de cada MMSI. Así no reaparecen escalas históricas cuyo destino fue uno de los tres puertos.

`anchored_elapsed_hours` se calcula como:

`último event_timestamp del buque - anchor_start_timestamp`

Es el equivalente en tiempo real de `actual_wait_hours = service_start_timestamp - anchor_start_timestamp` del Gold base.


In [0]:
latest_raw = positions.groupBy("mmsi").agg(F.max("event_timestamp").alias("_latest_raw_event_timestamp"))
w_current = Window.partitionBy("mmsi").orderBy(F.col("event_timestamp").desc())

current_position = (
    track.withColumn("_rn", F.row_number().over(w_current)).filter(F.col("_rn") == 1).drop("_rn")
    .join(latest_raw, "mmsi", "inner")
    .filter((F.col("event_timestamp") == F.col("_latest_raw_event_timestamp")) & F.col("inside_approach_zone") & (F.col("port_call_sequence") > 0))
    .select(
        *keys,
        F.col("event_timestamp").alias("current_event_timestamp"),
        F.col("distance_to_service_nm").alias("current_distance_to_service_nm"),
        F.col("speed_over_ground_knots").alias("current_speed_over_ground_knots"),
        F.col("navigation_status_code").alias("current_navigation_status_code"),
    )
)

current_calls = calls.join(current_position, keys, "inner").filter(F.col("service_start_timestamp").isNull())
current_calls = current_calls.withColumn("current_status", F.when(F.col("anchor_start_timestamp").isNotNull(), "ANCHORED").when(F.col("has_approach_evidence") == 1, "APPROACH").otherwise("OTHER")).filter(F.col("current_status").isin("APPROACH", "ANCHORED"))
current_calls = current_calls.withColumn("anchored_elapsed_hours", F.when(F.col("current_status") == "ANCHORED", (F.unix_timestamp("current_event_timestamp") - F.unix_timestamp("anchor_start_timestamp")) / 3600.0))


## 7. Baseline + modelo registrado

La mediana histórica se lee desde `waiting_avg_per_length`.

El modelo se carga siempre mediante el alias `@Champion`; no hay que copiar `run_id` ni modificar este notebook cuando se registre una versión nueva.


In [0]:
baseline = spark.table(baseline_table).select("destination_port_code", "vessel_length_band", "median_wait_hours")
current_calls = current_calls.join(F.broadcast(baseline), ["destination_port_code", "vessel_length_band"], "left")

client = MlflowClient()
model_version_info = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, registered_model_alias)
model_version = model_version_info.version
features_path = client.download_artifacts(model_version_info.run_id, "features.json")
with open(features_path, "r") as f: model_features = json.load(f)["features"]
model = mlflow.sklearn.load_model(MODEL_URI)
print("Modelo activo:", REGISTERED_MODEL_NAME, "| version:", model_version, "| features:", model_features)


Modelo activo: masterxyz002dbr.gold.vessel_jit_classifier | version: 1 | features: ['speed_over_ground_knots', 'navigation_status_code', 'destination_port_code']


## 8. Predicción JIT

Solo las tres variables definidas en `MODEL_FEATURES` se envían al clasificador.

Si un buque ya está fondeado y `anchored_elapsed_hours > median_wait_hours`, se marca `NO_JIT` por evidencia observada, independientemente de la salida del modelo.


In [0]:
score_pdf = current_calls.select("port_call_id", *model_features).toPandas()

if len(score_pdf):
    X_current = score_pdf[model_features]
    labels = model.predict(X_current).astype(int)
    probabilities = model.predict_proba(X_current)
    classes = list(getattr(model, "classes_", [0, 1]))
    jit_idx = classes.index(1)

    pred_pdf = score_pdf[["port_call_id"]].copy()
    pred_pdf["model_jit_label"] = labels
    pred_pdf["model_jit_probability"] = probabilities[:, jit_idx]
    predictions = spark.createDataFrame(pred_pdf)
else:
    predictions = spark.createDataFrame([], "port_call_id string, model_jit_label int, model_jit_probability double")

scored = current_calls.join(predictions, "port_call_id", "left")
scored = scored.withColumn("prediction_source", F.when((F.col("current_status") == "ANCHORED") & F.col("median_wait_hours").isNotNull() & (F.col("anchored_elapsed_hours") > F.col("median_wait_hours")), "ELAPSED_WAIT_RULE").otherwise("ML_MODEL"))
scored = scored.withColumn("predicted_jit_label", F.when(F.col("prediction_source") == "ELAPSED_WAIT_RULE", F.lit(0)).otherwise(F.col("model_jit_label")))
scored = scored.withColumn("predicted_jit_class", F.when(F.col("predicted_jit_label") == 1, "JIT").when(F.col("predicted_jit_label") == 0, "NO_JIT"))
scored = scored.withColumn("prediction_timestamp", F.current_timestamp()).withColumn("model_version", F.lit(str(model_version))).withColumn("model_uri", F.lit(MODEL_URI))


## 9. Gold final

Se mantienen los nombres del snapshot del Gold base (`event_timestamp`, `speed_over_ground_knots`, `navigation_status_code`, etc.).

Las columnas `current_*` describen el último AIS disponible y no son features del modelo. Esto permite distinguir claramente el momento usado para predecir del estado operativo actual.


In [0]:
final_df = scored.select(
    "port_call_id",
    "mmsi",
    "destination_port_code",
    "ship_type_category",
    "vessel_length_meters",
    "vessel_length_band",

    "current_status",
    "current_event_timestamp",
    F.round("current_distance_to_service_nm", 3).alias("current_distance_to_service_nm"),
    F.round("current_speed_over_ground_knots", 3).alias("current_speed_over_ground_knots"),
    "current_navigation_status_code",

    "event_timestamp",
    F.round("speed_over_ground_knots", 3).alias("speed_over_ground_knots"),
    F.round("course_over_ground_degrees", 3).alias("course_over_ground_degrees"),
    F.round("true_heading_degrees", 3).alias("true_heading_degrees"),
    "navigation_status_code",
    F.round("distance_to_service_nm", 3).alias("distance_to_service_nm"),

    "anchor_start_timestamp",
    F.round("anchored_elapsed_hours", 3).alias("anchored_elapsed_hours"),
    F.round("median_wait_hours", 3).alias("median_wait_hours"),

    "predicted_jit_label",
    "predicted_jit_class",
    F.round("model_jit_probability", 4).alias("model_jit_probability"),
    "prediction_source",
    "prediction_timestamp",
    "model_version",
    "model_uri",
)

write_delta_table(df=final_df, target_table=target_table, target_path=target_path)
print(f"[OK] Gold creada: {target_table}")


[OK] Gold creada: masterxyz002dbr.gold.vessel_jit_current_predictions


In [0]:
# ============================================================
# 10. RESUMEN
# ============================================================

result = spark.table(target_table)

display(result.agg(
    F.count("*").alias("vessels_to_monitor"),
    F.sum((F.col("current_status") == "APPROACH").cast("int")).alias("approach"),
    F.sum((F.col("current_status") == "ANCHORED").cast("int")).alias("anchored"),
    F.sum((F.col("predicted_jit_class") == "JIT").cast("int")).alias("predicted_jit"),
    F.sum((F.col("predicted_jit_class") == "NO_JIT").cast("int")).alias("predicted_no_jit"),
))

display(result.select(
    "port_call_id", "mmsi", "destination_port_code", "current_status",
    "current_event_timestamp", "event_timestamp", "anchor_start_timestamp",
    "anchored_elapsed_hours", "median_wait_hours",
    "speed_over_ground_knots", "navigation_status_code",
    "predicted_jit_class", "model_jit_probability", "prediction_source",
).orderBy(F.when(F.col("predicted_jit_class") == "NO_JIT", 0).otherwise(1), F.desc("anchored_elapsed_hours")))


vessels_to_monitor,approach,anchored,predicted_jit,predicted_no_jit
52,24,28,19,33


port_call_id,mmsi,destination_port_code,current_status,current_event_timestamp,event_timestamp,anchor_start_timestamp,anchored_elapsed_hours,median_wait_hours,speed_over_ground_knots,navigation_status_code,predicted_jit_class,model_jit_probability,prediction_source
6d2895ecef6ea99b015027f958079d339601e602aa451c9ec25d2b4b9cbe64a7,636092775,ESVLC,ANCHORED,2026-08-05T13:16:47.811585Z,null,2026-08-01T23:37:45.971285Z,85.651,2.12,null,null,NO_JIT,0.3091,ELAPSED_WAIT_RULE
a4eb2d6d9dec44aabb25646dec96f1bf2310ee49a68dc48152bc31e3890deebc,209575000,ESBCN,ANCHORED,2026-08-05T13:29:57.86833Z,null,2026-08-01T23:53:08.325946Z,85.614,4.9,null,null,NO_JIT,0.2704,ELAPSED_WAIT_RULE
646dfe0b5289c17c48eb042ba25dd4ebe626c94302b81d1794a9969c6eaf379e,219718000,ESVLC,ANCHORED,2026-08-05T13:27:44.860563Z,2026-08-02T17:41:48.344643Z,2026-08-02T17:51:57.309263Z,67.596,2.12,0.7,0,NO_JIT,0.4948,ELAPSED_WAIT_RULE
a43613365afdaf3d38fe17b53fdaae5640dba905c7b00e2f84d8dcda304e9c57,277550000,ESVLC,ANCHORED,2026-08-05T13:29:51.823196Z,2026-08-03T10:18:22.017626Z,2026-08-03T10:27:03.103387Z,51.047,29.48,1.1,0,NO_JIT,0.5021,ELAPSED_WAIT_RULE
b43e1b730b405543eb50ca7a62601c808134c129d71d432a385cbde0920da37c,636023084,ESBCN,ANCHORED,2026-08-05T13:25:13.195873Z,2026-08-03T11:21:13.276532Z,2026-08-03T17:38:51.817009Z,43.773,1.96,0.9,0,NO_JIT,0.4516,ELAPSED_WAIT_RULE
522129b131709c13cf573a5dee03d23666df887dc97493ef5b226aeaf4a8be40,636025763,ESBCN,ANCHORED,2026-08-05T13:05:11.625568Z,2026-08-04T13:39:14.204838Z,2026-08-04T13:47:14.444817Z,23.299,1.96,1.1,0,NO_JIT,0.4553,ELAPSED_WAIT_RULE
fb78bb31085a8ea69b3f60d88f1a11eab047504b50717d78ca684777116b8ffd,255993000,ESBCN,ANCHORED,2026-08-05T13:27:21.507526Z,2026-08-04T20:05:43.28625Z,2026-08-04T20:08:42.39852Z,17.311,4.9,0.6,1,NO_JIT,0.4385,ELAPSED_WAIT_RULE
65d070278e93cf50c83847b5904f26466829e4edcc1cfdd96a8e2b23c110025d,247418400,ESBCN,ANCHORED,2026-08-03T09:09:21.418692Z,null,2026-08-02T17:57:45.180126Z,15.193,4.9,null,null,NO_JIT,0.2704,ELAPSED_WAIT_RULE
eefb38d98db21cf6986a58acf62d7f5c082b541583ff1035a7af3dbf890d79e3,229665000,ESVLC,ANCHORED,2026-08-03T09:05:54.579257Z,null,2026-08-02T17:56:48.561908Z,15.152,27.21,null,null,NO_JIT,0.3091,ML_MODEL
9f043c33791ae3202ed5111b4ef5a50beb97e5023388d26607a89825676d4bae,256968000,ESVLC,ANCHORED,2026-08-03T09:07:36.768192Z,null,2026-08-02T17:58:37.917874Z,15.15,2.12,null,null,NO_JIT,0.3091,ELAPSED_WAIT_RULE


## Columnas de ML vs lectura operativa

**Features reales del modelo**
- `speed_over_ground_knots`
- `navigation_status_code`
- `destination_port_code`

**Contexto operativo**
- `event_timestamp`: snapshot usado para ML, con la misma semántica que el Gold base.
- `current_event_timestamp`: último AIS disponible.
- `anchor_start_timestamp`: inicio de fondeo detectado con la misma lógica del Gold base.
- `anchored_elapsed_hours`: horas transcurridas desde ese fondeo hasta el último AIS.
- `median_wait_hours`: baseline histórico por puerto + banda de eslora.
- `current_*`, rumbo y distancia: trazabilidad/lectura; no entran en el modelo.
